# Claussifier - Model Accuracy Evaluation

**Goal:** Evaluate all 3 trained models on the full LexGLUE `unfair_tos` test set and produce comparative visualizations.

**Models evaluated:**
1. **BERT** (`bert-base-uncased`) -- with optimized per-class thresholds
2. **LegalBERT** (`nlpaueb/legal-bert-base-uncased`) -- default 0.5 thresholds
3. **LegalBERT + Augmentation** (`nlpaueb/legal-bert-base-uncased`) -- trained on negation-augmented data, default 0.5 thresholds

**Visual outputs:**
- Per-class F1 / Precision / Recall grouped bar chart (3 models side by side)
- Multilabel confusion matrix heatmaps (one figure per model)

---

## 1. Setup & Installation

In [ ]:
!pip install -q datasets transformers torch scikit-learn matplotlib seaborn

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AutoTokenizer
from datasets import load_dataset, load_from_disk
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    multilabel_confusion_matrix
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import json
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Claussifier"
RESULTS_DIR = f"{PROJECT_DIR}/results/evaluation"

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Results will be saved to: {RESULTS_DIR}")

## 3. Load Datasets

In [ ]:
LABEL_NAMES = [
    "Limitation of liability",
    "Unilateral termination",
    "Unilateral change",
    "Content removal",
    "Contract by using",
    "Choice of law",
    "Jurisdiction",
    "Arbitration"
]
NUM_LABELS = len(LABEL_NAMES)

print("Loading standard LexGLUE unfair_tos dataset...")
standard_dataset = load_dataset("lex_glue", "unfair_tos")

standard_test_texts = standard_dataset['test']['text']
standard_test_labels = standard_dataset['test']['labels']

print(f"  Standard test set: {len(standard_test_texts)} examples")

AUGMENTED_DATA_PATH = f"{PROJECT_DIR}/data/unfair_tos_negation_augmented"

if os.path.exists(AUGMENTED_DATA_PATH):
    print("\nLoading augmented dataset...")
    augmented_dataset = load_from_disk(AUGMENTED_DATA_PATH)
    augmented_test_texts = augmented_dataset['test']['text']
    augmented_test_labels = augmented_dataset['test']['labels']
    print(f"  Augmented test set: {len(augmented_test_texts)} examples")
else:
    print(f"\nAugmented dataset not found at {AUGMENTED_DATA_PATH}")
    print("LegalBERT+Aug will be evaluated on the standard test set instead.")
    augmented_test_texts = standard_test_texts
    augmented_test_labels = standard_test_labels

## 4. Define Model Configurations

In [ ]:
BERT_THRESHOLDS_PATH = f"{PROJECT_DIR}/results/optimal_thresholds.json"

if os.path.exists(BERT_THRESHOLDS_PATH):
    with open(BERT_THRESHOLDS_PATH, 'r') as f:
        bert_thresholds = json.load(f)['optimal_thresholds']
    print(f"Loaded BERT optimized thresholds: {bert_thresholds}")
else:
    bert_thresholds = [0.5] * NUM_LABELS
    print("BERT optimized thresholds not found, using default 0.5")

MODEL_CONFIGS = [
    {
        'name': 'BERT',
        'model_dir': f"{PROJECT_DIR}/models/final_model",
        'test_texts': standard_test_texts,
        'test_labels': standard_test_labels,
        'thresholds': bert_thresholds,
    },
    {
        'name': 'LegalBERT',
        'model_dir': f"{PROJECT_DIR}/models/legalbert_model",
        'test_texts': standard_test_texts,
        'test_labels': standard_test_labels,
        'thresholds': [0.5] * NUM_LABELS,
    },
    {
        'name': 'LegalBERT+Aug',
        'model_dir': f"{PROJECT_DIR}/models/legalbert_with_augmentation",
        'test_texts': augmented_test_texts,
        'test_labels': augmented_test_labels,
        'thresholds': [0.5] * NUM_LABELS,
    },
]

print("\nModel configurations:")
for cfg in MODEL_CONFIGS:
    exists = os.path.isdir(cfg['model_dir'])
    print(f"  {cfg['name']}: {cfg['model_dir']} {'[FOUND]' if exists else '[MISSING]'}")
    print(f"    Test examples: {len(cfg['test_texts'])}")
    print(f"    Thresholds: {cfg['thresholds']}")

## 5. Evaluation Utilities

In [ ]:
class ClauseDataset(Dataset):
    """Custom Dataset for legal clause classification."""

    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label_list = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        label_tensor = torch.zeros(len(LABEL_NAMES))
        for lbl in label_list:
            label_tensor[lbl] = 1.0

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': label_tensor
        }


def run_inference(model, dataloader, device, thresholds):
    """Run batched inference and return raw probabilities and binary predictions."""
    model.eval()
    all_probs = []
    all_preds = []
    all_labels = []

    thresholds_tensor = torch.tensor(thresholds, device=device)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits)
            preds = (probs >= thresholds_tensor).float()

            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    return (
        np.concatenate(all_probs, axis=0),
        np.concatenate(all_preds, axis=0),
        np.concatenate(all_labels, axis=0),
    )


def compute_metrics(preds, labels, label_names):
    """Compute per-class and aggregate multilabel metrics."""
    per_class = []
    for i, name in enumerate(label_names):
        f1 = f1_score(labels[:, i], preds[:, i], zero_division=0)
        prec = precision_score(labels[:, i], preds[:, i], zero_division=0)
        rec = recall_score(labels[:, i], preds[:, i], zero_division=0)
        per_class.append({'Risk Category': name, 'F1': f1, 'Precision': prec, 'Recall': rec})

    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    macro_prec = precision_score(labels, preds, average='macro', zero_division=0)
    macro_rec = recall_score(labels, preds, average='macro', zero_division=0)

    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    micro_prec = precision_score(labels, preds, average='micro', zero_division=0)
    micro_rec = recall_score(labels, preds, average='micro', zero_division=0)

    cm = multilabel_confusion_matrix(labels, preds)

    return {
        'per_class': per_class,
        'macro_f1': macro_f1,
        'macro_precision': macro_prec,
        'macro_recall': macro_rec,
        'micro_f1': micro_f1,
        'micro_precision': micro_prec,
        'micro_recall': micro_rec,
        'confusion_matrices': cm,
    }


print("Evaluation utilities defined.")

## 6. Run Evaluation Loop

In [ ]:
BATCH_SIZE = 16
results = {}

for cfg in MODEL_CONFIGS:
    name = cfg['name']
    print(f"\n{'='*70}")
    print(f"Evaluating: {name}")
    print(f"{'='*70}")

    print(f"Loading model from {cfg['model_dir']}...")
    tokenizer = AutoTokenizer.from_pretrained(cfg['model_dir'])
    model = BertForSequenceClassification.from_pretrained(
        cfg['model_dir'],
        num_labels=NUM_LABELS,
    )
    model.to(DEVICE)
    model.eval()

    test_dataset = ClauseDataset(
        cfg['test_texts'], cfg['test_labels'], tokenizer, max_length=512
    )
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print(f"Test set: {len(test_dataset)} examples, {len(test_loader)} batches")

    probs, preds, labels = run_inference(model, test_loader, DEVICE, cfg['thresholds'])
    metrics = compute_metrics(preds, labels, LABEL_NAMES)

    results[name] = {
        'probs': probs,
        'preds': preds,
        'labels': labels,
        'metrics': metrics,
        'thresholds': cfg['thresholds'],
    }

    print(f"\n{name} Results:")
    print(f"  Macro  -- F1: {metrics['macro_f1']:.4f}  Precision: {metrics['macro_precision']:.4f}  Recall: {metrics['macro_recall']:.4f}")
    print(f"  Micro  -- F1: {metrics['micro_f1']:.4f}  Precision: {metrics['micro_precision']:.4f}  Recall: {metrics['micro_recall']:.4f}")
    print(f"\n  Per-class F1:")
    for pc in metrics['per_class']:
        print(f"    {pc['Risk Category']:<28s}  F1={pc['F1']:.4f}  P={pc['Precision']:.4f}  R={pc['Recall']:.4f}")

    del model, tokenizer
    torch.cuda.empty_cache()

print(f"\n{'='*70}")
print("All models evaluated.")
print(f"{'='*70}")

## 7. Visual Output 1: Per-Class Grouped Bar Chart

In [ ]:
model_names = list(results.keys())
metrics_list = ['F1', 'Precision', 'Recall']
colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)

x = np.arange(NUM_LABELS)
bar_width = 0.25

for ax_idx, metric_name in enumerate(metrics_list):
    ax = axes[ax_idx]

    for m_idx, m_name in enumerate(model_names):
        values = [pc[metric_name] for pc in results[m_name]['metrics']['per_class']]
        offset = (m_idx - 1) * bar_width
        bars = ax.bar(x + offset, values, bar_width, label=m_name, color=colors[m_idx], edgecolor='white', linewidth=0.5)

        for bar, val in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                f'{val:.2f}', ha='center', va='bottom', fontsize=6.5, fontweight='bold'
            )

    macro_key = f"macro_{metric_name.lower()}"
    for m_idx, m_name in enumerate(model_names):
        macro_val = results[m_name]['metrics'][macro_key]
        ax.axhline(y=macro_val, color=colors[m_idx], linestyle='--', alpha=0.5, linewidth=1)
        ax.text(
            NUM_LABELS - 0.5, macro_val + 0.01,
            f"{m_name} macro: {macro_val:.3f}",
            fontsize=7, color=colors[m_idx], ha='right', fontstyle='italic'
        )

    ax.set_ylabel(metric_name, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.12)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

axes[-1].set_xticks(x)
short_labels = [name.replace('Limitation of liability', 'Limitation\nof liability')
                     .replace('Unilateral termination', 'Unilateral\ntermination')
                     .replace('Unilateral change', 'Unilateral\nchange')
                     .replace('Content removal', 'Content\nremoval')
                     .replace('Contract by using', 'Contract\nby using')
                     .replace('Choice of law', 'Choice\nof law')
                for name in LABEL_NAMES]
axes[-1].set_xticklabels(short_labels, fontsize=9, ha='center')

fig.suptitle('Claussifier: Per-Class Model Comparison', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])

save_path = f"{RESULTS_DIR}/model_comparison_per_class.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved to {save_path}")

## 8. Visual Output 2: Multilabel Confusion Matrix Heatmaps

In [ ]:
for m_name in model_names:
    cm = results[m_name]['metrics']['confusion_matrices']

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    fig.suptitle(f'Confusion Matrices: {m_name}', fontsize=16, fontweight='bold', y=1.02)

    for i, (label, ax) in enumerate(zip(LABEL_NAMES, axes.flat)):
        tn, fp, fn, tp = cm[i].ravel()
        matrix = np.array([[tn, fp], [fn, tp]])

        sns.heatmap(
            matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Neg', 'Pred Pos'],
            yticklabels=['True Neg', 'True Pos'],
            ax=ax, cbar=False,
            annot_kws={'size': 12, 'fontweight': 'bold'}
        )
        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=8)

    plt.tight_layout()

    safe_name = m_name.lower().replace('+', '_plus_').replace(' ', '_')
    save_path = f"{RESULTS_DIR}/confusion_matrices_{safe_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved to {save_path}\n")

## 9. Summary Comparison Table & Export

In [ ]:
summary_rows = []
for m_name in model_names:
    m = results[m_name]['metrics']
    summary_rows.append({
        'Model': m_name,
        'Test Examples': len(results[m_name]['labels']),
        'Macro F1': round(m['macro_f1'], 4),
        'Macro Precision': round(m['macro_precision'], 4),
        'Macro Recall': round(m['macro_recall'], 4),
        'Micro F1': round(m['micro_f1'], 4),
        'Micro Precision': round(m['micro_precision'], 4),
        'Micro Recall': round(m['micro_recall'], 4),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print("=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
display(summary_df)

summary_path = f"{RESULTS_DIR}/model_comparison_summary.csv"
summary_df.to_csv(summary_path)
print(f"\nSaved summary to {summary_path}")

all_per_class = []
for m_name in model_names:
    for pc in results[m_name]['metrics']['per_class']:
        all_per_class.append({'Model': m_name, **pc})

per_class_df = pd.DataFrame(all_per_class)
per_class_path = f"{RESULTS_DIR}/model_comparison_per_class.csv"
per_class_df.to_csv(per_class_path, index=False)
print(f"Saved per-class results to {per_class_path}")

print("\nPer-class breakdown:")
display(per_class_df.pivot_table(index='Risk Category', columns='Model', values='F1').round(4))

---

### Files Generated

All outputs are saved to `{RESULTS_DIR}/evaluation/`:

| File | Description |
|---|---|
| `model_comparison_per_class.png` | Per-class F1/Precision/Recall grouped bar chart |
| `confusion_matrices_bert.png` | BERT confusion matrices (8 classes) |
| `confusion_matrices_legalbert.png` | LegalBERT confusion matrices |
| `confusion_matrices_legalbert_plus_aug.png` | LegalBERT+Aug confusion matrices |
| `model_comparison_summary.csv` | Aggregate metrics comparison |
| `model_comparison_per_class.csv` | Per-class metrics for all models |

### Notes

- BERT uses **optimized per-class thresholds** (0.65-0.70) from threshold optimization. LegalBERT and LegalBERT+Aug use default **0.5** thresholds.
- LegalBERT+Aug is evaluated on the **augmented test set** (1648 examples) to match its training conditions. BERT and LegalBERT use the standard test split (1607 examples).